# VFX Production Intelligence Dashboard — Time Entries Cleaning

Work through this table from top to bottom. The context and rules below are inherited from earlier milestones.

## What we already know

- **Business name:** Time Entries
- **Purpose:** Contains records for time entries used by the approved project analysis, including shot receiving the time, project receiving the time, and artist logging the time.
- **One row represents:** One row per logged work entry
- **Expected primary key:** `time_entry_id`

### Relationships
- shot_id → shots.shot_id
- project_id → projects.project_id
- artist_id → artists.artist_id

### Field rules from the approved data dictionary

| Field | Definition | Expected type | Null rule | Key role | Uniqueness | Allowed values / format | Cleaning expectation |
|---|---|---|---|---|---|---|---|
| `time_entry_id` | Unique identifier assigned to each logged work or timesheet entry. | Identifier text | No nulls allowed | Primary key | Required | TE-#######: uppercase TE, a hyphen, and seven digits. | Remove exact duplicate entries, quarantine conflicting repeated IDs, standardize formatting, and verify uniqueness and completeness. |
| `shot_id` | Identifier of the production shot receiving the logged work time. | Identifier text | No nulls allowed | Foreign key | Not required | PRJ-####_SQ###_SH####. | Trim and uppercase valid IDs, correct only confidently recoverable values, and quarantine blank, placeholder, or unresolved shot references. |
| `project_id` | Identifier of the project receiving the logged work time. | Identifier text | No nulls allowed | Foreign key | Not required | PRJ-####. | Trim and uppercase IDs, validate against projects.project_id, reconcile to the project inherited through shot_id, and quarantine unresolved contradictions. |
| `artist_id` | Identifier of the artist who logged the work time. | Identifier text | No nulls allowed | Foreign key | Not required | ART-###. | Trim and uppercase valid IDs, validate against artists.artist_id, and quarantine blank or unresolved references from utilization and cost analysis. |
| `work_date` | Calendar date on which the logged work was performed. | Date | No nulls allowed | Not a key | Not required | YYYY-MM-DD. | Parse valid formats, store as a true date, and quarantine placeholder or invalid values that cannot be corrected reliably. |
| `hours_logged` | Number of labor hours recorded for the individual timesheet entry. | Decimal | No nulls allowed | Not a key | Not required | Greater than 0 and no more than 24 hours. | Convert valid values to decimal, remove text suffixes, reject values <= 0 or > 24, and quarantine unresolved entries from hour-based KPIs. |
| `task_type` | Category describing the type of work performed during the time entry. | Category text | No nulls allowed | Not a key | Not required | Production; Review Fixes; Technical Setup; QC; R&D. | Trim whitespace and standardize to the five approved task-type labels. |
| `billable_flag` | Indicates whether the logged time is billable to the client. | Boolean | No nulls allowed | Not a key | Not required | Y = billable; N = non-billable. | Standardize affirmative values to Y and negative values to N; investigate unrecognized values. |
| `overtime_flag` | Indicates whether the time entry represents overtime work. | Boolean | No nulls allowed | Not a key | Not required | Y = overtime; N = regular time. | Standardize overtime-equivalent values to Y and regular-time equivalents to N; investigate unrecognized values. |
| `source_system` | Source application or export through which the time entry was recorded. | Category text | No nulls allowed | Not a key | Not required | Production Tracker; Timesheet Portal. | Trim whitespace and standardize to the two approved source-system labels. |
| `comment` | Optional free-form comment providing context about the logged work. | Text | Nulls allowed | Not a key | Not required | Free-form text, or null when no comment is recorded. | Trim populated text, convert empty strings to null, and preserve the original wording. |

### Known issues and approved decisions
- time_entry_id: The raw table contains 6,935 rows and 6,865 distinct IDs, producing 70 repeated rows.
- time_entry_id: time_entry_id remains the intended primary key. Repeated values are duplicate raw fact rows that must be removed or resolved.
- shot_id: The raw table contains 45 shot-key issues, including 12 blanks, MISSING_SHOT placeholders, lowercase IDs, and unresolved references.
- shot_id: shot_id remains a required foreign key. Entries without a resolvable shot cannot safely support shot-level labor analysis.
- project_id: The raw table contains project-key issues, blanks, case variants, nonexistent values, and 147 records whose project ID disagrees with the related shot.
- project_id: project_id remains a required foreign key and must agree with both the Projects table and the project derived from shot_id.
- artist_id: The raw table contains artist-key issues, including 14 blanks, whitespace/case variants, and unresolved artist references.
- artist_id: artist_id remains a required foreign key. Entries without a resolvable artist cannot safely support utilization or labor-cost metrics.
- work_date: The raw text field contains 110 mixed or invalid date issues, including TBD, alternate formats, and impossible dates.
- work_date: The field is logically a required date; malformed source values explain the text type and require cleaning.
- hours_logged: The raw text field contains 52 type or range issues, including 8h, seven, blanks, zero, negative values, and entries above 24 hours.
- hours_logged: The field is expected to be a positive numeric duration not exceeding 24 hours; malformed and out-of-range source values require remediation.
- billable_flag: The raw field includes Y, N, Yes, 1, TRUE, and Billable.
- billable_flag: The source stores one boolean concept with several text encodings. The cleaned field will use only Y and N.
- overtime_flag: The raw field includes Y, N, No, OT, FALSE, and 0.
- overtime_flag: The source stores one boolean concept with several text encodings. The cleaned field will use only Y and N.
- comment: Missing comments are expected; repeated common comments do not by themselves indicate duplicate entries.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r'C:/Users/Dan/Documents/MEGA/Dev/GitHub/career-accelerator/projects/project-01-vfx-production-intelligence')
TABLE_NAME = 'time_entries'
RAW_PATH = PROJECT_DIR / r'data/raw/csv/raw_time_entries.csv'
PROCESSED_PATH = PROJECT_DIR / r'data/processed/csv/time_entries.csv'
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

if RAW_PATH.suffix.lower() == '.csv':
    raw_df = pd.read_csv(RAW_PATH)
elif RAW_PATH.suffix.lower() == '.parquet':
    raw_df = pd.read_parquet(RAW_PATH)
else:
    raise ValueError(f'Add the appropriate pandas reader for {RAW_PATH.suffix}')

clean_df = raw_df.copy()
print(f'{TABLE_NAME}: {len(raw_df):,} raw rows, {len(raw_df.columns)} columns')
raw_df.head()


## 1. Profile the raw table

- Confirm the source row count and column names.
- Measure missing values by field.
- Check exact duplicate rows.
- Test uniqueness and nulls for `time_entry_id`.
- Review observed categories and parsing problems.
- Compare findings with the dictionary rules above before changing data.

In [ ]:
# Write the profiling checks for this table here.
# Keep the outputs that justify your cleaning decisions.


## 2. Apply the approved cleaning plan

Transform `clean_df` without modifying `raw_df`. Follow the field-level expectations above. Document any treatment that differs from the approved dictionary.

In [ ]:
# Write this table's cleaning transformations here.
# Example structure only: clean_df = clean_df.copy()


## 3. Validate the processed result

- Required columns are still present.
- Expected logical types can be produced consistently.
- Required fields do not contain unresolved nulls.
- Allowed values and formats match the dictionary.
- Invalid negative, out-of-range, or impossible values are resolved or documented.
- `time_entry_id` is non-null and unique.
- Foreign-key and relationship exceptions are measured and documented.

In [ ]:
# Write the before-and-after validation checks here.
# The checks should fail visibly when an unresolved issue remains.


## 4. Export the reviewed table

After validation, save the reviewed result to `data/processed/csv/time_entries.csv`. The Data Cleaning Studio will discover and validate the file.

In [ ]:
# Run only after the table has passed your validation checks.
clean_df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved {len(clean_df):,} rows to {PROCESSED_PATH}')


## Cleaning summary

<!-- Describe what changed, why each important decision was appropriate, how many records were affected, and any remaining exception that a later milestone must know about. -->
